# 11.3 检索策略本节涵盖：Top-K, 重排序, 自适应检索, 多跳检索

In [ ]:
import torchimport torch.nn as nnimport torch.nn.functional as Ftorch.manual_seed(42)class CrossEncoderReranker(nn.Module):    def __init__(self, dim=64):        super().__init__()        self.net = nn.Sequential(nn.Linear(dim*2, 64), nn.ReLU(), nn.Linear(64, 1))    def forward(self, query, doc):        return self.net(torch.cat([query, doc], dim=-1)).squeeze(-1)reranker = CrossEncoderReranker()query = torch.randn(1, 64)docs = torch.randn(20, 64)# Stage 1: Fast retrieval (bi-encoder)q_emb = F.normalize(query, dim=-1)d_embs = F.normalize(docs, dim=-1)retrieval_scores = (q_emb @ d_embs.T).squeeze(0)top_k_indices = retrieval_scores.topk(5).indices# Stage 2: Reranking (cross-encoder)candidates = docs[top_k_indices]query_expanded = query.expand(5, -1)rerank_scores = reranker(query_expanded, candidates)final_ranking = rerank_scores.argsort(descending=True)print('=== Two-Stage Retrieval + Reranking ===')print(f'Stage 1: Retrieved top-5 from {len(docs)} docs (fast bi-encoder)')print(f'Stage 2: Reranked with cross-encoder (accurate but slower)')print(f'Final ranking: {final_ranking.tolist()}')print()print('=== Adaptive Retrieval ===')print('Not all queries need retrieval!')print('Train classifier: [needs_retrieval] vs [no_retrieval]')print('Only trigger RAG when model lacks knowledge')print()print('=== Multi-Hop Retrieval ===')print('Round 1: Retrieve initial docs for query')print('Round 2: Generate follow-up query based on Round 1 results')print('Round 3: Retrieve more docs with refined query')print('Continue until sufficient information gathered')print('Example: "Who is the spouse of the director of Inception?"')print('  Hop 1: "Director of Inception" -> Christopher Nolan')print('  Hop 2: "Spouse of Christopher Nolan" -> Emma Thomas')